# ORF Finder & Protein Translation Pipeline

This notebook identifies Open Reading Frames (ORFs) in DNA sequences,
translates them into protein sequences, and visualizes ORF statistics.

### Features:
- FASTA parsing
- Forward + reverse strand analysis
- ORF detection across 3 frames
- DNA → protein translation
- Data visualization

Imports

In [5]:
import matplotlib.pyplot as plt

Codon Table as Dictionary

In [6]:
CODON_TABLE = {
    "ATA":"I","ATC":"I","ATT":"I","ATG":"M",
    "ACA":"T","ACC":"T","ACG":"T","ACT":"T",
    "AAC":"N","AAT":"N","AAA":"K","AAG":"K",
    "AGC":"S","AGT":"S","AGA":"R","AGG":"R",
    "CTA":"L","CTC":"L","CTG":"L","CTT":"L",
    "CCA":"P","CCC":"P","CCG":"P","CCT":"P",
    "CAC":"H","CAT":"H","CAA":"Q","CAG":"Q",
    "CGA":"R","CGC":"R","CGG":"R","CGT":"R",
    "GTA":"V","GTC":"V","GTG":"V","GTT":"V",
    "GCA":"A","GCC":"A","GCG":"A","GCT":"A",
    "GAC":"D","GAT":"D","GAA":"E","GAG":"E",
    "GGA":"G","GGC":"G","GGG":"G","GGT":"G",
    "TCA":"S","TCC":"S","TCG":"S","TCT":"S",
    "TTC":"F","TTT":"F","TTA":"L","TTG":"L",
    "TAC":"Y","TAT":"Y","TAA":"*","TAG":"*",
    "TGC":"C","TGT":"C","TGA":"*","TGG":"W",
}

Outputs DNA sequence into readable fasta file

In [7]:
def read_fasta(filename):
    sequences = {}
    header = None

    with open(filename) as file:
        for line in file:
            line = line.strip()

            if line.startswith(">"):
                header = line[1:]
                sequences[header] = ""
            else:
                sequences[header] += line.upper()

    return sequences

Generates reverse strand sequence for later comprehensive ORF search (6 frames)

In [8]:
def reverse_complement(sequence):
    complement = {"A": "T", "T": "A", "C": "G", "G": "C"}
    return "".join(complement[base] for base in reversed(sequence))

Detects position of ORF based on location of start codons and respective stop codons

In [10]:
def find_orfs(sequence):
    start_codon = "ATG"
    stop_codons = {"TAA", "TAG", "TGA"}

    orfs = []

    for frame in range(3):

        for i in range(frame, len(sequence) - 2, 3):

            codon = sequence[i:i+3]

            if codon == start_codon:

                for j in range(i, len(sequence) - 2, 3):

                    codon = sequence[j:j+3]

                    if codon in stop_codons:

                        start = i
                        end = j + 3

                        orf_seq = sequence[start:end]
                        protein = translate_dna(orf_seq)
                        aa_length = len(protein)

                        orfs.append((start, end, frame, aa_length, protein))
                        break

    return orfs

Translates codons within ORF into amino acids based on codon dictionary

In [11]:
def translate_dna(dna_sequence):
    protein = ""

    for i in range(0, len(dna_sequence) - 2, 3):
        codon = dna_sequence[i:i+3]
        amino_acid = CODON_TABLE.get(codon, "X")

        if amino_acid == "*":
            break

        protein += amino_acid

    return protein

Generates the output (sequence, strand, frame, start, stop, AA count, Protein) based on the original and complementary strand.

Frame (0-2) depending on the location of the first AA in the DNAsequence. Significant as it determines the reading of the following codons (3 base pairs) in the entire sequence.

In [17]:
def analyze_fasta(input_file, output_file="orf_results.tsv"):

    sequences = read_fasta(input_file)

    with open(output_file, "w") as out:

        out.write("Sequence\tStrand\tFrame\tStart\tEnd\tAminoAcids\tProtein\n")

        for name, seq in sequences.items():
            analyze_sequence(name, seq, out)

    print(f"Results saved to {output_file}")

ORF results separated into AA length and frame components for plot processing

In [19]:
def load_orf_results(filename):
    orf_lengths = []
    orf_frames = []

    with open(filename) as f:
        next(f)

        for line in f:
            parts = line.strip().split("\t")

            aa_length = int(parts[5])
            frame = int(parts[2])

            orf_lengths.append(aa_length)
            orf_frames.append(frame)

    return orf_lengths, orf_frames

ORF Length Distribution Plot

In [21]:
def plot_orf_lengths(plot_lengths):

    plt.figure()
    plt.hist(plot_lengths, bins=20, edgecolor='black')

    plt.xlabel("Amino Acid Length")
    plt.ylabel("Frequency")
    plt.title("ORF Length Distribution")

    plt.yscale("log")

    plt.savefig("orf_length_distribution.png")
    plt.show()

ORF Frame Distribution Plot (ORFs vs. Frame)

In [22]:
def plot_frame_distribution(plot_frames):

    frame_counts = {0: 0, 1: 0, 2: 0}

    for f in plot_frames:
        frame_counts[f] += 1

    plt.figure()
    plt.bar(frame_counts.keys(), frame_counts.values())

    plt.xlabel("Reading Frame")
    plt.ylabel("Number of ORFs")
    plt.title("ORF Distribution by Frame")

    plt.savefig("frame_distribution.png")
    plt.show()